<a href="https://colab.research.google.com/github/PrudhviNallagatla/Advanced-Recognition-of-License-Plates-ARLP/blob/main/src/ARLP-MobileNetV2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
!mkdir -p /content/dataset
!unzip -q /content/drive/MyDrive/Elearnmarkets/dataset.zip -d /content

In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
from glob import glob

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import albumentations as A

tf.random.set_seed(42)
np.random.seed(42)

In [3]:
from glob import glob

import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, Input
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

import albumentations as A

tf.random.set_seed(42)
np.random.seed(42)

### Data Augmentation

In [5]:
IMAGE_SIZE = 224
BATCH_SIZE = 16

def extract_box(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()
    # Get image dimensions
    size = root.find('size')
    width = float(size.find('width').text)
    height = float(size.find('height').text)

    # Get bounding box
    bndbox = root.find('.//bndbox')
    xmin = float(bndbox.find('xmin').text)
    ymin = float(bndbox.find('ymin').text)
    xmax = float(bndbox.find('xmax').text)
    ymax = float(bndbox.find('ymax').text)

    return [xmin, ymin, xmax, ymax], width, height

def get_data(data_dir):
    images = []
    boxes = []

    xml_files = glob(os.path.join(data_dir, '*.xml'))
    for xml_file in xml_files:
        img_file = xml_file.replace('.xml', '.jpg')
        if not os.path.exists(img_file):
            img_file = xml_file.replace('.xml', '.png') # Fallback to png

        if os.path.exists(img_file):
            # Load and convert image to RGB
            img = cv2.imread(img_file)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

            # Extract coordinates
            box, w, h = extract_box(xml_file)

            images.append(img)

            # We keep the raw [xmin, ymin, xmax, ymax] for albumentations
            boxes.append([box])

    return images, boxes

# TRAIN_DIR = 'dataset/ARLP Datasets ver-2/train/'
TRAIN_DIR = 'dataset/ARLP Datasets ver-2/train/'
TEST_DIR = 'dataset/ARLP Datasets ver-2/test/'
# TEST_DIR = 'dataset/ARLP Datasets ver-2/test/'

train_images, train_boxes = get_data(TRAIN_DIR)
val_images, val_boxes = get_data(TEST_DIR)

print(f"Loaded {len(train_images)} training images and {len(val_images)} validation images.")

Loaded 1000 training images and 142 validation images.


### Albumentations generator

In [6]:
class BoundingBoxDataGenerator(tf.keras.utils.Sequence):
    def __init__(self, images, boxes, batch_size, augment=False):
        self.images = images
        self.boxes = boxes
        self.batch_size = batch_size

        # Albumentations pipeline
        if augment:
            self.transform = A.Compose([
                A.Resize(IMAGE_SIZE, IMAGE_SIZE),
                A.HorizontalFlip(p=0.5),
                A.RandomBrightnessContrast(p=0.2),
                A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=10, p=0.5, border_mode=cv2.BORDER_CONSTANT)
            ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))
        else:
            # Only resize for validation
            self.transform = A.Compose([
                A.Resize(IMAGE_SIZE, IMAGE_SIZE)
            ], bbox_params=A.BboxParams(format='pascal_voc', label_fields=['labels']))

    def __len__(self):
        return int(np.ceil(len(self.images) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_images = self.images[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_boxes = self.boxes[idx * self.batch_size:(idx + 1) * self.batch_size]

        X = np.zeros((len(batch_images), IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.float32)
        y = np.zeros((len(batch_images), 4), dtype=np.float32)

        for i in range(len(batch_images)):
            transformed = self.transform(
                image=batch_images[i],
                bboxes=batch_boxes[i],
                labels=[1] # Dummy label
            )

            # Normalize image to [0, 1]
            X[i] = transformed['image'] / 255.0

            # If box disappeared after cropping/augmenting, use a default centered box to prevent NaN
            if len(transformed['bboxes']) == 0:
                box = [0, 0, IMAGE_SIZE, IMAGE_SIZE]
            else:
                box = transformed['bboxes'][0]

            # Normalize bounding box to [0, 1] relative to the RESIZED image
            y[i] = [box[0]/IMAGE_SIZE, box[1]/IMAGE_SIZE, box[2]/IMAGE_SIZE, box[3]/IMAGE_SIZE]

        return X, y

train_gen = BoundingBoxDataGenerator(train_images, train_boxes, BATCH_SIZE, augment=True)
val_gen = BoundingBoxDataGenerator(val_images, val_boxes, BATCH_SIZE, augment=False)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


### MobileNetV2 Architecture

In [7]:
def build_model():
    # Load pretrained MobileNetV2
    base_model = MobileNetV2(weights='imagenet', include_top=False, input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3))

    # Freeze the backbone (it acts as a feature extractor)
    base_model.trainable = False

    # Add our custom regression head
    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.2)(x)
    x = Dense(128, activation='relu')(x)

    # 4 output coordinates (xmin, ymin, xmax, ymax) normalized between 0 and 1
    predictions = Dense(4, activation='sigmoid', name='bounding_box')(x)

    model = Model(inputs=base_model.input, outputs=predictions, name="ARLP_MobileNetV2")
    return model

model = build_model()
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='mse')
model.summary()

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "ARLP_MobileNetV2"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,619,332 (9.99 MB)

 Trainable params: 361,348 (1.38 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

### training

In [8]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ModelCheckpoint('ARLP-MobileNetV2.keras', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, verbose=1)
]

history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=100,
    callbacks=callbacks
)

Epoch 1/100


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step - loss: 0.0437
Epoch 1: val_loss improved from None to 0.02553, saving model to ARLP-MobileNetV2.keras

Epoch 1: finished saving model to ARLP-MobileNetV2.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 53s 535ms/step - loss: 0.0369 - val_loss: 0.0255 - learning_rate: 1.0000e-04
Epoch 2/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 44ms/step - loss: 0.0309
Epoch 2: val_loss improved from 0.02553 to 0.02382, saving model to ARLP-MobileNetV2.keras

Epoch 2: finished saving model to ARLP-MobileNetV2.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 58ms/step - loss: 0.0296 - val_loss: 0.0238 - learning_rate: 1.0000e-04
Epoch 3/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step - loss: 0.0275
Epoch 3: val_loss improved from 0.02382 to 0.02290, saving model to ARLP-MobileNetV2.keras

Epoch 3: finished saving model to ARLP-MobileNetV2.keras
63/63 ━━━━━━━━━━━━━━━━━━━━ 4s 69ms/step - loss: 0.0278 - val_loss: 0.0229 - learning_rate: 1.0000e-04
Epoch 4/100
63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step - loss